# Módulo 7 · Clase 4 — Clasificación y métricas

**Curso de Inteligencia Artificial — Nivel básico**

---

## ¿Qué vamos a ver hoy?

| # | Tema |
|---|------|
| 1 | Regresión Logística: ¿cómo predice categorías? |
| 2 | Matriz de confusión |
| 3 | Accuracy, Precision, Recall y F1 |
| 4 | Curva ROC y AUC |
| 5 | Clases desbalanceadas |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, roc_auc_score
)

np.random.seed(42)
print("Librerías listas")

---
## 1. Regresión Logística

```
Regresión lineal → predice un número continuo (-∞, +∞)
Regresión logística → predice una PROBABILIDAD [0, 1] usando la función sigmoide:

  P(y=1 | x) = 1 / (1 + e^(-z))    donde z = β₀ + β₁x₁ + ... + βₙxₙ

Clasificación: si P(y=1) ≥ umbral (default 0.5) → clase 1, si no → clase 0

¿Por qué no usar regresión lineal para clasificar?
  - Puede predecir valores fuera de [0,1]
  - Una muestra extrema puede distorsionar la frontera de decisión
```

In [ ]:
# Dataset: predecir si un correo es spam
np.random.seed(42)
n = 1000
n_palabras_spam = np.random.exponential(15, n)
n_enlaces       = np.random.poisson(3, n)
urgencia        = np.random.uniform(0, 10, n)

z = -4 + 0.15*n_palabras_spam + 0.5*n_enlaces + 0.3*urgencia
prob_spam = 1 / (1 + np.exp(-z))
spam = (np.random.rand(n) < prob_spam).astype(int)

df = pd.DataFrame({
    'palabras_spam': n_palabras_spam,
    'n_enlaces':     n_enlaces,
    'urgencia':      urgencia,
    'spam':          spam
})
print(f"Dataset: {n} correos, {spam.sum()} spam ({spam.mean()*100:.1f}%)")

X = df[['palabras_spam', 'n_enlaces', 'urgencia']]
y = df['spam']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

modelo = LogisticRegression(random_state=42)
modelo.fit(X_tr_s, y_tr)

print(f"\nCoeficientes:")
for feat, coef in zip(X.columns, modelo.coef_[0]):
    print(f"  {feat:20}: {coef:+.4f}")
print(f"Intercepto: {modelo.intercept_[0]:.4f}")

# Probabilidades
probas = modelo.predict_proba(X_te_s)
print(f"\nPrimeras 5 predicciones:")
print(f"{'Prob No Spam':>14} {'Prob Spam':>12} {'Predicción':>12} {'Real':>8}")
for i in range(5):
    pred = modelo.predict([X_te_s[i]])[0]
    print(f"{probas[i,0]:>14.4f} {probas[i,1]:>12.4f} {pred:>12} {y_te.iloc[i]:>8}")

# Función sigmoide
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
z_vals = np.linspace(-6, 6, 100)
sigmoid = 1 / (1 + np.exp(-z_vals))
axes[0].plot(z_vals, sigmoid, 'b-', lw=2)
axes[0].axhline(0.5, color='red', ls='--', lw=1, alpha=0.7, label='umbral=0.5')
axes[0].axvline(0, color='gray', ls='-', lw=0.5)
axes[0].fill_between(z_vals, 0, sigmoid, where=z_vals>0, alpha=0.1, color='green')
axes[0].set_title('Función sigmoide: z → P(spam)')
axes[0].set_xlabel('z = β₀ + Σβᵢxᵢ'); axes[0].set_ylabel('Probabilidad')
axes[0].legend()

axes[1].scatter(X_te_s[:,0], probas[:,1], c=y_te.values, cmap='bwr', alpha=0.5, s=20)
axes[1].axhline(0.5, color='k', ls='--', lw=2, label='Umbral 0.5')
axes[1].set_xlabel('Palabras spam (escalado)'); axes[1].set_ylabel('P(spam)')
axes[1].set_title('Probabilidad predicha vs feature principal')
axes[1].legend()
plt.tight_layout(); plt.show()

---
## 2. Matriz de confusión

```
                   PREDICCIÓN
              │  Negativo  │  Positivo  │
    ──────────┼────────────┼────────────┤
    Negativo  │     TN     │     FP     │  ← No Spam real
    ──────────┼────────────┼────────────┤
    Positivo  │     FN     │     TP     │  ← Spam real

TN (True Negative):  correo normal, predicho normal ✓
FP (False Positive): correo normal, predicho spam ✗ (falsa alarma)
FN (False Negative): spam, predicho normal ✗ (spam no detectado)
TP (True Positive):  spam, predicho spam ✓
```

In [ ]:
y_pred = modelo.predict(X_te_s)
cm = confusion_matrix(y_te, y_pred)

print("Matriz de confusión:")
print(f"                Predicho No Spam | Predicho Spam")
print(f"Real No Spam:   TN={cm[0,0]:3d}              FP={cm[0,1]:3d}")
print(f"Real Spam:      FN={cm[1,0]:3d}              TP={cm[1,1]:3d}")

# Visualización como heatmap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels(['No Spam', 'Spam'])
axes[0].set_yticklabels(['No Spam', 'Spam'])
axes[0].set_xlabel('Predicción'); axes[0].set_ylabel('Real')
for i in range(2):
    for j in range(2):
        texto = f"{cm[i,j]}\n{'TN FP FN TP'.split()[i*2+j]}"
        axes[0].text(j, i, f"{cm[i,j]}", ha='center', va='center', fontsize=14,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Matriz de Confusión')

# Normalizada
cm_norm = cm / cm.sum(axis=1, keepdims=True)
im2 = axes[1].imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels(['No Spam', 'Spam'])
axes[1].set_yticklabels(['No Spam', 'Spam'])
axes[1].set_xlabel('Predicción'); axes[1].set_ylabel('Real')
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm_norm[i,j]:.2%}", ha='center', va='center', fontsize=13,
                    color='white' if cm_norm[i,j] > 0.5 else 'black')
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('Matriz de Confusión (normalizada)')
plt.tight_layout(); plt.show()

---
## 3. Métricas de clasificación

```
ACCURACY:   (TP + TN) / (TP + TN + FP + FN)
  → % de predicciones correctas
  ✗ Engañosa con clases desbalanceadas: 95% accuracy si el 95% es clase 0

PRECISION:  TP / (TP + FP)
  → De los que predije positivos, ¿cuántos son realmente positivos?
  → Importa cuando los FP son caros: sistema anti-spam → no quiero bloquear emails buenos

RECALL:     TP / (TP + FN)
  → De todos los positivos reales, ¿cuántos detecté?
  → Importa cuando los FN son caros: diagnóstico cáncer → no quiero dejar sin detectar

F1:         2 × (Precision × Recall) / (Precision + Recall)
  → Media armónica de Precision y Recall
  → Equilibrio entre ambas
  → Útil con clases desbalanceadas

Tradeoff Precision-Recall: subir uno baja el otro (controlado por el umbral de clasificación)
```

In [ ]:
tn, fp, fn, tp = cm.ravel()

print("=== Métricas calculadas manualmente ===")
acc   = (tp + tn) / (tp + tn + fp + fn)
prec  = tp / (tp + fp)
rec   = tp / (tp + fn)
f1    = 2 * prec * rec / (prec + rec)
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}  (de los que detecté como spam, el {prec*100:.1f}% lo eran)")
print(f"Recall:    {rec:.4f}  (detecto el {rec*100:.1f}% del spam real)")
print(f"F1:        {f1:.4f}")

print("\n=== Verificación con sklearn ===")
print(f"accuracy_score:  {accuracy_score(y_te, y_pred):.4f}")
print(f"precision_score: {precision_score(y_te, y_pred):.4f}")
print(f"recall_score:    {recall_score(y_te, y_pred):.4f}")
print(f"f1_score:        {f1_score(y_te, y_pred):.4f}")

print("\n=== Informe completo ===")
print(classification_report(y_te, y_pred, target_names=['No Spam', 'Spam']))

# Efecto del umbral sobre Precision/Recall
probas_spam = modelo.predict_proba(X_te_s)[:, 1]
umbrales = np.arange(0.1, 0.95, 0.05)
precisiones, recalls, f1s = [], [], []

for u in umbrales:
    y_pred_u = (probas_spam >= u).astype(int)
    if y_pred_u.sum() == 0: continue
    precisiones.append(precision_score(y_te, y_pred_u, zero_division=0))
    recalls.append(recall_score(y_te, y_pred_u))
    f1s.append(f1_score(y_te, y_pred_u))

fig, ax = plt.subplots(figsize=(10, 4))
u_plot = umbrales[:len(precisiones)]
ax.plot(u_plot, precisiones, 'b-o', ms=4, label='Precision')
ax.plot(u_plot, recalls,     'r-o', ms=4, label='Recall')
ax.plot(u_plot, f1s,         'g-o', ms=4, label='F1')
ax.axvline(0.5, color='gray', ls='--', alpha=0.7, label='Umbral por defecto')
ax.set_xlabel('Umbral de clasificación'); ax.set_ylabel('Métrica')
ax.set_title('Tradeoff Precision-Recall según umbral')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4. Curva ROC y AUC

```
ROC (Receiver Operating Characteristic):
  Representa el tradeoff entre:
    - TPR (True Positive Rate) = Recall = TP/(TP+FN)  [eje Y]
    - FPR (False Positive Rate)= FP/(FP+TN)           [eje X]
  para todos los umbrales posibles.

AUC (Area Under the Curve):
  - AUC = 1.0: modelo perfecto
  - AUC = 0.5: modelo aleatorio (línea diagonal)
  - AUC < 0.5: peor que aleatorio (invertir predicciones)

¿Cuándo usar AUC en vez de Accuracy?
  Cuando las clases están desbalanceadas o el umbral es ajustable.
  AUC mide la calidad del ranking, no del umbral.
```

In [ ]:
fpr, tpr, thresholds = roc_curve(y_te, probas_spam)
auc = roc_auc_score(y_te, probas_spam)

# Comparar con baseline aleatorio
np.random.seed(0)
prob_random = np.random.rand(len(y_te))
fpr_r, tpr_r, _ = roc_curve(y_te, prob_random)
auc_r = roc_auc_score(y_te, prob_random)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'LogisticRegression (AUC={auc:.4f})')
axes[0].plot(fpr_r, tpr_r, 'r--', lw=1, label=f'Aleatorio (AUC={auc_r:.4f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, alpha=0.5)
axes[0].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[0].set_xlabel('FPR (Tasa de Falsos Positivos)')
axes[0].set_ylabel('TPR (Recall)')
axes[0].set_title('Curva ROC')
axes[0].legend()

# Umbral óptimo (maximizar TPR - FPR)
youden_idx = np.argmax(tpr - fpr)
umbral_opt = thresholds[youden_idx]
axes[0].scatter(fpr[youden_idx], tpr[youden_idx], color='red', s=100, zorder=5,
               label=f'Óptimo (umbral={umbral_opt:.3f})')
axes[0].legend(fontsize=8)

print(f"AUC: {auc:.4f}")
print(f"Umbral óptimo (Youden): {umbral_opt:.4f}")

# Comparar métricas con umbral 0.5 vs óptimo
for umbral, nombre in [(0.5, 'Umbral 0.5'), (umbral_opt, f'Umbral óptimo ({umbral_opt:.3f})')]:
    y_pred_u = (probas_spam >= umbral).astype(int)
    axes[1].bar(
        [f'{nombre}\nPrecision', f'{nombre}\nRecall', f'{nombre}\nF1'],
        [precision_score(y_te, y_pred_u), recall_score(y_te, y_pred_u), f1_score(y_te, y_pred_u)],
        alpha=0.7
    )

axes[1].set_ylim(0, 1); axes[1].set_title('Métricas por umbral')
axes[1].set_ylabel('Valor')
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

---
## 5. Clases desbalanceadas

In [ ]:
# Dataset muy desbalanceado: 5% de fraude
np.random.seed(42)
n_total = 1000
n_fraude = 50  # solo 5%

X_norm   = np.random.randn(n_total - n_fraude, 5)
X_fraude = np.random.randn(n_fraude, 5) + 1.5  # ligeramente diferente
X_imb    = np.vstack([X_norm, X_fraude])
y_imb    = np.array([0]*(n_total-n_fraude) + [1]*n_fraude)

X_tr, X_te, y_tr, y_te = train_test_split(X_imb, y_imb, test_size=0.3, random_state=42, stratify=y_imb)

print("=== Problema con clases desbalanceadas ===")

# Baseline: predice siempre clase mayoritaria
y_baseline = np.zeros(len(y_te), dtype=int)
print(f"Baseline (predice siempre 0):")
print(f"  Accuracy:  {accuracy_score(y_te, y_baseline):.4f}  ← ¡parece bueno!")
print(f"  F1(fraude): {f1_score(y_te, y_baseline, pos_label=1):.4f}  ← ¡desastre!")
print(f"  Recall(fraude): {recall_score(y_te, y_baseline, pos_label=1, zero_division=0):.4f}")

# LogisticRegression sin balancear
m_sin = LogisticRegression(random_state=42).fit(X_tr, y_tr)
y_pred_sin = m_sin.predict(X_te)

# LogisticRegression con class_weight='balanced'
m_bal = LogisticRegression(class_weight='balanced', random_state=42).fit(X_tr, y_tr)
y_pred_bal = m_bal.predict(X_te)

print(f"\nLogisticRegression SIN balanceo:")
print(classification_report(y_te, y_pred_sin, target_names=['Normal','Fraude']))

print(f"LogisticRegression CON class_weight='balanced':")
print(classification_report(y_te, y_pred_bal, target_names=['Normal','Fraude']))

print("\nOtras estrategias para datos desbalanceados:")
print("  1. class_weight='balanced' en el modelo (más sencillo)")
print("  2. Oversampling de la clase minoritaria (SMOTE)")
print("  3. Undersampling de la clase mayoritaria")
print("  4. Ajustar el umbral de clasificación")
print("  5. Usar F1 o AUC como métrica, NO accuracy")

---
## Práctica

### Ejercicio 1 — Titanic con Logistic Regression

**Enunciado:** Predice la supervivencia en el Titanic:

```python
import seaborn as sns
df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked']].dropna()
```

1. Preprocesa: codifica sex y embarked, escala age y fare
2. Entrena LogisticRegression
3. Imprime accuracy, precision, recall y F1
4. Visualiza la matriz de confusión
5. ¿Cuál métrica te parece más importante aquí? ¿Por qué?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 1</summary>

```python
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

df = sns.load_dataset('titanic')[['survived','pclass','sex','age','fare','embarked']].dropna()

# 1. Preprocesar
df['sex_num'] = (df['sex'] == 'female').astype(int)
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)
cols_escalar = ['age', 'fare']

X = df.drop(['survived','sex'], axis=1)
y = df['survived']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
sc = StandardScaler()
X_tr[cols_escalar] = sc.fit_transform(X_tr[cols_escalar])
X_te[cols_escalar] = sc.transform(X_te[cols_escalar])

# 2. Entrenar
m = LogisticRegression(max_iter=1000, random_state=42).fit(X_tr, y_tr)
y_pred = m.predict(X_te)

# 3. Métricas
print(f"Accuracy:  {accuracy_score(y_te, y_pred):.4f}")
print(f"Precision: {precision_score(y_te, y_pred):.4f}")
print(f"Recall:    {recall_score(y_te, y_pred):.4f}")
print(f"F1:        {f1_score(y_te, y_pred):.4f}")
print(classification_report(y_te, y_pred, target_names=['No sobrevivió','Sobrevivió']))

# 4. Matriz de confusión
cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(5,4))
plt.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2):
        plt.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=14)
plt.xticks([0,1],['No','Sí']); plt.yticks([0,1],['No','Sí'])
plt.xlabel('Predicho'); plt.ylabel('Real')
plt.title('Matriz de confusión'); plt.tight_layout(); plt.show()

# 5. Reflexión
print("\nEn el Titanic, si el modelo predice 'sobrevivió' pero no sobrevivió,")
print("la consecuencia es menor que predecir 'no sobrevivió' cuando sí sobrevivió")
print("→ El Recall de la clase 'Sobrevivió' es la métrica más importante aquí.")
```
</details>

---
### Ejercicio 2 — Efecto del umbral de clasificación

**Enunciado:** Sistema de detección de tumores malignos:

```python
from sklearn.datasets import load_breast_cancer
data = load_breast_cancer()
```

1. Entrena LogisticRegression y obtén las probabilidades de la clase positiva (maligno)
2. Prueba umbrales: [0.3, 0.4, 0.5, 0.6, 0.7]
3. Para cada umbral: Precision, Recall, F1
4. ¿Qué umbral elegirías si el coste de un FN (cáncer no detectado) es muy alto?
5. Traza la curva ROC y calcula AUC

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 2</summary>

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             roc_curve, roc_auc_score)
import matplotlib.pyplot as plt
import numpy as np

data = load_breast_cancer()
X, y = data.data, data.target  # 1=benigno, 0=maligno
# Invertir: positivo = maligno (0)
y_inv = 1 - y

X_tr, X_te, y_tr, y_te = train_test_split(X, y_inv, test_size=0.2, random_state=42, stratify=y_inv)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(max_iter=10000).fit(X_tr_s, y_tr)
probas = m.predict_proba(X_te_s)[:, 1]  # P(maligno)

print(f"{'Umbral':>8} {'Precision':>10} {'Recall':>10} {'F1':>10}")
for u in [0.3, 0.4, 0.5, 0.6, 0.7]:
    y_pred = (probas >= u).astype(int)
    p = precision_score(y_te, y_pred, zero_division=0)
    r = recall_score(y_te, y_pred)
    f = f1_score(y_te, y_pred)
    print(f"{u:>8.1f} {p:>10.4f} {r:>10.4f} {f:>10.4f}")

# 4.
print("\nSi FN (cáncer no detectado) es muy costoso → maximizar RECALL.")
print("→ Usar umbral bajo (0.3): detectamos más cánceres aunque haya más FP (biopsias innecesarias).")

# 5. ROC
fpr, tpr, _ = roc_curve(y_te, probas)
auc = roc_auc_score(y_te, probas)
plt.figure(figsize=(7,5))
plt.plot(fpr, tpr, lw=2, label=f'AUC = {auc:.4f}')
plt.plot([0,1],[0,1],'k--', alpha=0.5)
plt.xlabel('FPR'); plt.ylabel('TPR (Recall)')
plt.title('Curva ROC — Detección de tumor maligno')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
```
</details>

---
### Ejercicio 3 — Clasificación multiclase

**Enunciado:** LogisticRegression también funciona para múltiples clases (Iris tiene 3):

1. Carga Iris y entrena LogisticRegression con `multi_class='auto'`
2. Imprime el classification_report para las 3 clases
3. Visualiza la matriz de confusión (3×3)
4. ¿Qué clase confunde más el modelo? ¿Por qué tiene sentido biológicamente?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 3</summary>

```python
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

iris = datasets.load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(max_iter=1000, random_state=42)
m.fit(X_tr_s, y_tr)
y_pred = m.predict(X_te_s)

print(classification_report(y_te, y_pred, target_names=iris.target_names))

cm = confusion_matrix(y_te, y_pred)
plt.figure(figsize=(6,5))
plt.imshow(cm, cmap='Blues')
plt.xticks(range(3), iris.target_names, rotation=30)
plt.yticks(range(3), iris.target_names)
plt.xlabel('Predicho'); plt.ylabel('Real')
for i in range(3):
    for j in range(3):
        color = 'white' if cm[i,j] > cm.max()/2 else 'black'
        plt.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=13, color=color)
plt.colorbar(); plt.title('Matriz de confusión Iris'); plt.tight_layout(); plt.show()

print("\nVersicolor y Virginica se confunden más entre sí:")
print("Son las dos especies más similares morfológicamente.")
print("Setosa es perfectamente separable (diferente especie sin solapamiento).")
```
</details>

---
### Ejercicio 4 — Calcular métricas desde cero

**Enunciado:** Sin usar sklearn.metrics, implementa las cuatro métricas:

```python
y_real = np.array([1,0,1,1,0,1,0,0,1,1,0,1,0,0,1])
y_pred = np.array([1,0,1,0,0,1,1,0,1,1,0,0,0,1,1])
```

1. Calcula TP, TN, FP, FN
2. Calcula Accuracy, Precision, Recall, F1 manualmente
3. Verifica con sklearn
4. ¿Cuál de las 4 métricas te da más información útil en este caso?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 4</summary>

```python
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_real = np.array([1,0,1,1,0,1,0,0,1,1,0,1,0,0,1])
y_pred = np.array([1,0,1,0,0,1,1,0,1,1,0,0,0,1,1])

# 1. Componentes de la matriz de confusión
TP = np.sum((y_pred == 1) & (y_real == 1))
TN = np.sum((y_pred == 0) & (y_real == 0))
FP = np.sum((y_pred == 1) & (y_real == 0))
FN = np.sum((y_pred == 0) & (y_real == 1))
print(f"TP={TP}, TN={TN}, FP={FP}, FN={FN}")

# 2. Métricas
acc   = (TP + TN) / len(y_real)
prec  = TP / (TP + FP)
rec   = TP / (TP + FN)
f1    = 2 * prec * rec / (prec + rec)
print(f"\nAccuracy (manual):  {acc:.4f}")
print(f"Precision (manual): {prec:.4f}")
print(f"Recall (manual):    {rec:.4f}")
print(f"F1 (manual):        {f1:.4f}")

# 3. Verificar
print(f"\nAccuracy (sklearn):  {accuracy_score(y_real, y_pred):.4f}")
print(f"Precision (sklearn): {precision_score(y_real, y_pred):.4f}")
print(f"Recall (sklearn):    {recall_score(y_real, y_pred):.4f}")
print(f"F1 (sklearn):        {f1_score(y_real, y_pred):.4f}")

# 4. Interpretación
print("\nInterpretación:")
print(f"  De 15 ejemplos, 8 son positivos y 7 negativos → clases casi balanceadas.")
print(f"  Accuracy={acc:.0%}: el modelo acierta el {acc:.0%} de los casos.")
print(f"  F1={f1:.4f}: equilibrio entre precisión y recall.")
print("  Aquí F1 es la métrica más informativa por su equilibrio.")
```
</details>

---
### Ejercicio 5 (reto) — Detector de fraude con umbral dinámico

**Enunciado:** Tienes un sistema de detección de fraude. El coste de un FN (fraude no detectado) es 500€ y el coste de un FP (transacción legítima bloqueada) es 5€.

1. Genera un dataset de fraude (2% fraude): `np.random.seed(5); X = np.random.randn(2000,10); y = np.random.choice([0,1], 2000, p=[0.98, 0.02])`
2. Entrena LogisticRegression con `class_weight='balanced'`
3. Prueba umbrales de 0.1 a 0.9 y calcula el **coste total** para cada uno: `coste = FN*500 + FP*5`
4. Visualiza coste vs umbral
5. ¿Cuál es el umbral que minimiza el coste? ¿Coincide con el umbral de máximo F1?

In [ ]:
# ✏️ TU CÓDIGO AQUÍ


<details>
<summary>💡 Ver solución — Ejercicio 5</summary>

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.preprocessing import StandardScaler

np.random.seed(5)
X = np.random.randn(2000, 10)
y = np.random.choice([0,1], 2000, p=[0.98, 0.02])
print(f"Fraudes: {y.sum()} ({y.mean()*100:.1f}%)")

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s  = sc.transform(X_te)

m = LogisticRegression(class_weight='balanced', max_iter=1000).fit(X_tr_s, y_tr)
probas = m.predict_proba(X_te_s)[:,1]

umbrales = np.arange(0.05, 0.96, 0.05)
costes, f1s = [], []
for u in umbrales:
    y_pred = (probas >= u).astype(int)
    if y_pred.sum() == 0: continue
    tn, fp, fn, tp = confusion_matrix(y_te, y_pred).ravel()
    costes.append(fn*500 + fp*5)
    f1s.append(f1_score(y_te, y_pred))

u_plot = umbrales[:len(costes)]
u_min_coste = u_plot[np.argmin(costes)]
u_max_f1    = u_plot[np.argmax(f1s)]

fig, axes = plt.subplots(1, 2, figsize=(14,4))
axes[0].plot(u_plot, costes, 'r-o', ms=4)
axes[0].axvline(u_min_coste, color='blue', ls='--', label=f'Óptimo coste={u_min_coste:.2f}')
axes[0].set_title('Coste económico por umbral'); axes[0].set_xlabel('Umbral')
axes[0].set_ylabel('Coste (€)'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(u_plot, f1s, 'g-o', ms=4)
axes[1].axvline(u_max_f1, color='blue', ls='--', label=f'Óptimo F1={u_max_f1:.2f}')
axes[1].set_title('F1 por umbral'); axes[1].set_xlabel('Umbral')
axes[1].set_ylabel('F1'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()
print(f"Umbral mínimo coste: {u_min_coste:.2f} (coste={min(costes):,.0f}€)")
print(f"Umbral máximo F1:    {u_max_f1:.2f}")
print("\nNo coinciden necesariamente — F1 ignora los costes asimétricos.")
print("En producción: optimizar la métrica de negocio, no la métrica estadística.")
```
</details>

---
## Resumen

```python
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)

m = LogisticRegression(class_weight='balanced').fit(X_tr, y_tr)
y_pred = m.predict(X_te)
probas = m.predict_proba(X_te)[:, 1]  # probabilidad clase positiva

# Métricas
print(classification_report(y_te, y_pred))
auc = roc_auc_score(y_te, probas)

# Umbral personalizado
y_pred_custom = (probas >= 0.3).astype(int)

# Elegir métrica según el contexto:
# Clases balanceadas:    Accuracy o F1
# Clases desbalanceadas: F1, AUC-ROC
# FN muy costoso:        Recall
# FP muy costoso:        Precision
```

## ¿Qué viene en la próxima clase?
En la **Clase 5** aprendemos los **Árboles de Decisión** y **Random Forest**, modelos mucho más potentes que la regresión logística para la mayoría de problemas.